# 🧪 W5-D6 概念实验：Prompt 对照实验的正确姿势

> 配套阅读：`ima/第5周-Day6-推理Prompt实验.md`（四种策略、控制变量、温度与采样的理论在那边）
>
> 本 notebook 回答四个问题：
> 1. 为什么 Prompt 实验**必须固定随机种子**？换一批题结论会不会翻转？
> 2. 准确率-延迟-Token 成本的三角里，谁是 Pareto 最优？
> 3. 同样的 6 分提升：**配对检验** vs 非配对检验，结论为什么完全不同？
> 4. 怎么写"安全"的自动评分函数 + 可复现实验日志？

## 实验 1：种子就是"同一批题"——可复现是一切的前提

模拟 4 种策略（zero-shot / few-shot / CoT / CoT+自洽）在 50 道题上的表现。
seed 同时决定题目难度与模型采样。验证两件事：固定种子结果逐位可复现；换种子，准确率会漂移。

In [ ]:
import numpy as np

STRATS = {
    "zero-shot": dict(p=0.55, lat=120,  tok=380),
    "few-shot":  dict(p=0.62, lat=210,  tok=700),
    "CoT":       dict(p=0.71, lat=650,  tok=1400),
    "CoT+自洽":  dict(p=0.78, lat=3400, tok=8000),
}
N_Q = 50

def run_once(strategy, seed):
    """seed 固定题目难度与采样 → 完全可复现的一次'实验'"""
    r = np.random.default_rng(seed)
    q_difficulty = r.random(N_Q)                      # 题目本身的难度（种子固定）
    cfg = STRATS[strategy]
    correct = r.random(N_Q) < cfg["p"] * (0.5 + q_difficulty)
    latency = r.normal(cfg["lat"], cfg["lat"] * 0.2, N_Q).clip(1)
    tokens = r.normal(cfg["tok"], cfg["tok"] * 0.15, N_Q).clip(1)
    return correct, latency, tokens

a = run_once("CoT", seed=0)
b = run_once("CoT", seed=0)
print("固定种子跑两遍，逐题结果完全一致:", np.array_equal(a[0], b[0]))
print()
print("只换种子（= 换一批 50 题），CoT 的准确率：")
for seed in range(5):
    c, _, _ = run_once("CoT", seed)
    print(f"  seed={seed} → {c.mean():.0%}")
print("小样本 + 不固定种子 = 想要多结论就有多结论。实验报告必须写：题集、种子、prompt 版本。")

## 实验 2：准确率-延迟-Token 三角与 Pareto 前沿

每种策略跑 20 个种子取均值：横轴延迟，纵轴准确率，点大小 = Token 花销。
Pareto 前沿 = 没有任何别的策略"又快又准"地压过它。预算线该切在前沿上。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

acc_s, lat_s, tok_s = {}, {}, {}
for s in STRATS:
    accs, ls, ts = [], [], []
    for seed in range(20):
        c, l, t = run_once(s, seed)
        accs.append(c.mean()); ls.append(l.mean()); ts.append(t.mean())
    acc_s[s], lat_s[s], tok_s[s] = np.mean(accs), np.mean(ls), np.mean(ts)

names = list(STRATS)
pts = list(zip([lat_s[s] for s in names], [acc_s[s] for s in names], names))
front = sorted([p for p in pts if not any(q[0] <= p[0] and q[1] >= p[1] and q is not p for q in pts)])

plt.figure(figsize=(7, 4.4))
for lat, acc, name in pts:
    on_front = any(f[2] == name for f in front)
    plt.scatter(lat, acc * 100, s=tok_s[name] / 15, alpha=0.6,
                color="#fb8500" if on_front else "#8ecae6")
    plt.annotate(name, (lat, acc * 100), fontsize=9,
                 xytext=(6, 4), textcoords="offset points")
plt.plot([f[0] for f in front], [f[1] * 100 for f in front], "r--", lw=1, alpha=0.6)
plt.xscale("log")
plt.xlabel("平均延迟 ms（log）"); plt.ylabel("准确率 (%)")
plt.title("成本-准确率三角：橙点=Pareto 最优；点越大 Token 越贵")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Pareto 前沿:", " → ".join(f[2] for f in front))
print("被支配的策略不是'没用'：低价值场景它就是性价比答案——先说预算，再选点。")

## 实验 3：配对 vs 非配对——同一个提升，两种结论

200 道题上比较 zero-shot 与 CoT（本题模拟里 CoT 只会"救题"不会"坑题"，净提升约 +7pp）。
- 非配对：把 400 个 0/1 打乱重分两组 → 检验力低（题目难度噪声淹没差异）
- 配对：只在**同一道题内**翻转 A/B 标签 → 难度被抵消，只看"谁救了谁"

用置换检验算两种 p 值。

In [ ]:
import numpy as np

rng = np.random.default_rng(8)
n = 200
difficulty = rng.random(n)
zero = rng.random(n) < 0.55 * (0.5 + difficulty)
cot = zero | (rng.random(n) < 0.15)          # CoT 把一部分做错的题救回来（≈+7pp）

def perm_unpaired(x, y, n_perm=5000):
    """打乱重分（难度信息被破坏）"""
    r = np.random.default_rng(1)
    obs = x.mean() - y.mean()
    pool = np.concatenate([x, y]).astype(float)
    cnt = 0
    for _ in range(n_perm):
        r.shuffle(pool)
        if pool[:n].mean() - pool[n:].mean() >= obs:
            cnt += 1
    return obs, cnt / n_perm

def perm_paired(x, y, n_perm=5000):
    """只在题内翻转符号（难度作为配对信息保留）"""
    r = np.random.default_rng(1)
    obs = x.mean() - y.mean()
    d = x.astype(float) - y.astype(float)
    signs = r.random((n_perm, len(d))) < 0.5
    null = np.where(signs, d, -d).mean(axis=1)
    return obs, (null >= obs).mean()

obs, p_u = perm_unpaired(cot.astype(float), zero.astype(float))
_, p_p = perm_paired(cot.astype(float), zero.astype(float))
print(f"观察到的提升：{obs:+.1%}")
print(f"非配对置换检验 p ≈ {p_u:.3f} → {'显著' if p_u < 0.05 else '不显著（结论：看不出差别）'}")
print(f"配对置换检验   p ≈ {p_p:.5f} → {'显著' if p_p < 0.05 else '不显著'}")
print()
print("同一份数据、同一个问题，配对设计把'题目难度'这个最大噪声源控住了。")
print("评测两套 Prompt：同一批题、同一顺序、只换 prompt——这是控制变量的全部意义。")

## 实验 4：安全的评分函数 + 可复现的实验日志

自动评分的铁律：**绝不 eval 模型输出**，只做受限解析；
实验日志带 run_id / prompt 版本 / 种子，保证三个月后还能复现。

In [ ]:
import csv, io, re

def extract_number(text):
    """安全解析：只抽第一个数字，绝不执行模型输出"""
    m = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(m.group()) if m else None

def score(pred_text, gold, tol=1e-6):
    v = extract_number(pred_text)
    if v is None:
        return 0
    return int(abs(v - gold) <= max(tol, abs(gold) * tol))

runs = [
    ("q1", "CoT",       "所以答案是 16 元",        16),
    ("q1", "zero-shot", "大概是 16 吧",            16),
    ("q2", "CoT",       "5×3=15，3×2=6，21-5=16", 16),   # 陷阱：第一个数字是中间值
    ("q3", "zero-shot", "我不知道",                 16),
]
for qid, strat, resp, gold in runs:
    print(f"{qid} [{strat:9s}] 抽到数字={extract_number(resp)!s:>5}  score={score(resp, gold)}")
print("注意 q2：抽'第一个数字'会被中间步骤骗 → 评分函数应先要求'答案：X'格式再解析。")
print()

buf = io.StringIO()
w = csv.writer(buf)
w.writerow(["run_id", "question_set", "prompt_version", "seed", "correct", "latency_ms_avg", "tokens_avg"])
for i in range(3):
    c, l, t = run_once("CoT", seed=i)
    w.writerow([f"run{i:03d}", "gsm8k-mini", "cot-v2", i, int(c.sum()), round(l.mean(), 1), round(t.mean())])
print("实验日志（可直接落库）：")
print(buf.getvalue())

## 结论

- 不固定种子/题集的 Prompt 实验结论 = 噪声（实验 1）
- 策略选择 = 在 Pareto 前沿上按预算取点（实验 2）
- 配对设计把题目难度控住，小样本也能检出几分提升（实验 3）
- 评分函数永不 eval 输出；日志带版本与种子（实验 4）

→ 深入阅读：`ima/第5周-Day6-推理Prompt实验.md`（实验报告写法、CSV 导出规范）